# **Preparing Dependancies**

In [1]:
# =========================================================
# Persiapan library & device  (jalankan di Google Colab GPU T4)
# =========================================================
!pip install -q --upgrade diffusers transformers accelerate

import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline, StableDiffusionInpaintPipeline

# Pilih device otomatis: GPU (Colab T4) kalau tersedia, kalau tidak CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device aktif :", DEVICE)
if DEVICE == "cuda":
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 65.6 MB/s eta 0:00:00


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device aktif : cuda
GPU          : Tesla T4
VRAM (GB)    : 15.6


# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [2]:
# ---------------------------------------------------------
# Load pipeline Text-to-Image (Stable Diffusion v1.5)
# ---------------------------------------------------------
# Catatan model: repo asli "runwayml/stable-diffusion-v1-5" sudah di-takedown
# dari Hugging Face (organisasi runwayml delisted sejak 2024), jadi tidak bisa
# diunduh lagi. Kita pakai MIRROR RESMI yang isinya identik bit-per-bit:
#   stable-diffusion-v1-5/stable-diffusion-v1-5
SD_TXT2IMG = "stable-diffusion-v1-5/stable-diffusion-v1-5"

txt2img = StableDiffusionPipeline.from_pretrained(SD_TXT2IMG, torch_dtype=DTYPE)
txt2img = txt2img.to(DEVICE)
txt2img.safety_checker = None   # subjek astronot aman -> matikan agar tak ada blackout

print("Pipeline text-to-image siap dipakai.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Pipeline text-to-image siap dipakai.


## **Generate Image**

In [ ]:
# ---------------------------------------------------------
# K1 - generate_simple_image() : versi standar (prompt, negative_prompt, seed)
# ---------------------------------------------------------
def generate_simple_image(prompt, negative_prompt, seed):
    """Text-to-image standar (guidance & steps default) -> hasil kartun 2D."""
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out = txt2img(prompt=prompt, negative_prompt=negative_prompt, generator=gen)
    return out.images[0]

# REVISI penolakan_1 (K1) - v3 (setelah 2x cek visual di Colab): prompt dibuat
# MINIMAL. Percobaan sebelumnya memakai "planet earth ... starry sky background"
# + "full body, wide shot, flat 2d" -> malah bikin astronot chibi berdiri di
# "bola" (simple) DAN astronot hilang jadi lanskap kosong (advanced). Penyebab:
# frasa angkasa (planet/starry sky) menggeser komposisi jadi vista luar angkasa.
# Solusi: frasa bumi dibuat ringkas ("earth in the background") supaya bulan tetap
# jadi PERMUKAAN datar tempat astronot berdiri (target image-3), astronot tetap
# subjek utama. Prompt WAJIB sama dengan generate_advanced_image().
MOON_PROMPT = ("a lone astronaut standing on the moon surface, "
               "earth in the background, cartoon style")

# Negative prompt WAJIB (persis dari instruksi soal) -> menekan hasil realistis/3D.
# Justru pada fungsi SIMPLE inilah negative ini paling penting (jaga gaya kartun).
NEGATIVE = ("photorealistic, realistic, photograph, 3d render, messy, blurry, "
            "low quality, bad art, ugly, sketch, grainy, unfinished, "
            "chromatic aberration")

SEED_K1 = 222
img_simple = generate_simple_image(MOON_PROMPT, NEGATIVE, SEED_K1)
img_simple

## **Generate Image with Hyperparameter Configuration**

In [ ]:
# ---------------------------------------------------------
# K1 - generate_advanced_image() : + guidance_scale & num_inference_steps
# ---------------------------------------------------------
def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_steps):
    """Text-to-image dengan kontrol guidance_scale & jumlah inference step."""
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out = txt2img(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=gen,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    )
    return out.images[0]

# Target advanced (image-4) = versi 3D / REALISTIS, bukan kartun. Karena PROMPT
# wajib SAMA dengan simple (yang mengandung "cartoon style"), gaya dibalik lewat
# dua hal yang boleh berbeda antar-fungsi:
#   1) negative_prompt di sini justru MENOLAK "cartoon / flat / illustration /
#      painting" -> kata "cartoon" pada prompt jadi dibatalkan, model jatuh ke
#      distribusi default SD1.5 (banyak foto astronot Apollo) = realistis.
#   2) guidance_scale TINGGI (12.0) supaya inti prompt "astronaut on the moon"
#      dominan, + num_inference_steps 50 untuk detail tajam.
# Reviewer mengizinkan menyesuaikan negative prompt & parameter pada fungsi ini.
NEGATIVE_ADVANCED = ("cartoon, flat, 2d, illustration, drawing, painting, anime, "
                     "comic, cel shading, bold outline, low quality, blurry, "
                     "deformed, bad anatomy")

img_advanced = generate_advanced_image(
    prompt=MOON_PROMPT,                  # SAMA dengan simple (syarat soal)
    negative_prompt=NEGATIVE_ADVANCED,   # beda: menolak kartun -> memaksa realistis
    seed=SEED_K1,
    guidance_scale=12.0,
    num_inference_steps=50,
)
img_advanced

## **Guidance Scale Comparison**

### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah:**   
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, kesesuaian dengan prompt, dan variasi visual yang terlihat."*

*   **Gambar dengan "Scale" Tinggi:**   
*"Jelaskan perbedaan yang terlihat dibandingkan guidance scale rendah, terutama pada detail gambar dan kedekatannya dengan prompt."*

## **Inference Steps Comparison**

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**  
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, ketajaman, serta kemungkinan munculnya noise atau artefak."*
*   **Gambar dengan "Step" Tinggi:**  
*"Jelaskan perbedaan yang terlihat dibandingkan step rendah, terutama pada detail gambar, kehalusan hasil, dan stabilitas visual."*

## **Batch Inference from One Prompt**

## **Load Scheduler**

### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DPM++ Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DDIM Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*

# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Base + Refiner Image Generation**

## **Inpainting**

### **Load Model Inpainting**

In [5]:
# ---------------------------------------------------------
# K2 - Load pipeline khusus Inpainting (SD Inpainting v1.5)
# ---------------------------------------------------------
# Sama seperti model text-to-image, repo runwayml sudah delisted -> pakai mirror resmi.
SD_INPAINT = "stable-diffusion-v1-5/stable-diffusion-inpainting"

inpaint = StableDiffusionInpaintPipeline.from_pretrained(SD_INPAINT, torch_dtype=DTYPE)
inpaint = inpaint.to(DEVICE)
inpaint.safety_checker = None

print("Pipeline inpainting siap dipakai.")

model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Pipeline inpainting siap dipakai.


### **Manual Masking**

In [ ]:
# ---------------------------------------------------------
# K2 - Masking MANUAL (hardcode area, pendekatan trial & error)
# ---------------------------------------------------------
# Base yang di-edit = hasil generate_advanced_image (astronot REALISTIS di bulan,
# biasanya di TENGAH). Satelit rusak ditempel di sisi KANAN astronot supaya objek
# punya ruang & TIDAK menimpa astronot -- meniru komposisi image-6 (astronot kiri
# + satelit kanan). Aturan mask: PUTIH (255) = area diganti objek, HITAM (0) = tetap.
base_image = img_advanced
W, H = base_image.size

# Kotak area edit (proporsi gambar). Mask di sisi KANAN (x0 0.56) supaya tidak
# menimpa astronot yang di tengah, tapi tetap cukup lebar untuk satelit yang besar.
x0 = int(W * 0.56)   # kanan astronot -> tanah kosong
y0 = int(H * 0.40)
x1 = int(W * 0.98)
y1 = int(H * 0.90)

# Bangun mask lewat array numpy: 0 di seluruh kanvas, 255 di dalam kotak.
mask_arr = np.zeros((H, W), dtype=np.uint8)
mask_arr[y0:y1, x0:x1] = 255
mask_manual = Image.fromarray(mask_arr, mode="L")

# Pratinjau posisi: overlay merah semi-transparan di area yang akan di-inpaint.
# >>> WAJIB CEK: kalau kotak merah menyentuh badan astronot, NAIKKAN x0 (mis. 0.60)
#     lalu jalankan ulang sel ini sebelum lanjut ke Generate. <<<
preview = base_image.convert("RGBA")
overlay = Image.new("RGBA", (W, H), (255, 0, 0, 90))
preview.paste(overlay, (0, 0), mask_manual)
print(f"Ukuran gambar (W,H) = ({W},{H})")
print(f"Kotak mask (x0,y0,x1,y1) = ({x0},{y0},{x1},{y1})")
preview.convert("RGB")

### **Generate**

In [ ]:
# ---------------------------------------------------------
# K2 - inpaint_engine(image, mask, prompt) : isi area mask dengan objek baru
# ---------------------------------------------------------
def inpaint_engine(image, mask, prompt):
    """Inpainting: menambahkan satelit rusak di area mask.

    REVISI penolakan_1 (K2): reviewer menilai satelit belum muncul jelas.
    Perbaikan: prompt lebih menegaskan UKURAN objek + guidance_scale & steps
    TINGGI (CFG 20, 60 step) supaya objek dipaksa muncul (bukan sekadar tanah).
    negative_prompt menolak "tanah kosong" agar area mask tidak diisi ulang
    permukaan bulan polos. Seed = 9 sesuai ketentuan soal K2.
    """
    gen = torch.Generator(device=DEVICE).manual_seed(9)
    out = inpaint(
        prompt=prompt,
        negative_prompt=("empty, bare ground, plain lunar surface, flat sand, "
                         "smooth terrain, nothing, cartoon, painting, "
                         "blurry, low quality, deformed"),
        image=image,
        mask_image=mask,
        generator=gen,
        guidance_scale=20.0,
        num_inference_steps=60,
    )
    return out.images[0]

# Prompt satelit menegaskan objek BESAR & rinci (badan logam, sayap panel surya,
# antena, kaki pendarat) + gaya realistis supaya menyatu dengan base advanced yang
# realistis -> mendekati image-4 (target). Ukuran ditegaskan ("large", "huge",
# "filling the mid-ground") sesuai saran reviewer soal besar/kecil objek.
SATELLITE_PROMPT = ("a large broken satellite spacecraft crashed on the lunar surface, "
                    "huge metallic body filling the mid-ground, shattered solar panel wings, "
                    "bent antenna dish, scattered metal debris, exposed mechanical parts, "
                    "multi-legged landing gear, highly detailed mechanical structure, "
                    "sharp focus, realistic scale, photorealistic, cinematic lighting")

img_inpaint = inpaint_engine(base_image, mask_manual, SATELLITE_PROMPT)
img_inpaint

## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

### **Masking with Segmentation Model**

### **Generate**

## **Outpainting**

### **Prepare the Canvas**

### **Generate**

## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

### **Generate**